In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA 0 — Setup de entorno y extracción del dataset               ║
# ║  Detecta PROJECT_ROOT automáticamente y descomprime luna16.7z      ║
# ║                                                                      ║
# ║  Estructura del archivo:                                            ║
# ║    luna16.7z extrae todos los .npy PLANOS en la raíz               ║
# ║    (sin subcarpetas train/val/test) + candidates_V2.csv            ║
# ║                                                                      ║
# ║  Coloca luna16.7z en: <PROJECT_ROOT>/datasets/luna16/              ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os
import subprocess
from pathlib import Path

# ── Detección automática de PROJECT_ROOT ────────────────────────────
_cwd = Path(os.getcwd()).resolve()
if _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "notebooks").exists() and (_cwd / "src").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "notebooks").exists() and (_cwd.parent / "src").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd  # fallback: directorio actual
print(f"PROJECT_ROOT → {PROJECT_ROOT}")

# ── Crear carpeta del dataset ────────────────────────────────────────
LUNA_DIR = PROJECT_ROOT / "datasets" / "luna16"
LUNA_DIR.mkdir(parents=True, exist_ok=True)

# ── Verificar si ya está extraído ───────────────────────────────────
# Marcador de extracción: el CSV debe estar en la raíz (estructura plana)
_marker = LUNA_DIR / "candidates_V2.csv"
if _marker.exists():
    print("✓ Dataset LUNA16 ya extraído — saltando descompresión")
else:
    _archive = LUNA_DIR / "luna16.7z"
    if not _archive.exists():
        raise FileNotFoundError(
            f"\n{'━'*60}\n"
            f"  Archivo no encontrado: {_archive}\n"
            f"  Coloca luna16.7z en:\n"
            f"    {LUNA_DIR}\n"
            f"{'━'*60}"
        )
    _size_gb = _archive.stat().st_size / 1e9
    print(f"Extrayendo {_archive.name} ({_size_gb:.1f} GB) → {LUNA_DIR} ...")
    print("  (puede tardar varios minutos según el hardware)")
    _result = subprocess.run(
        ["7z", "x", str(_archive), f"-o{LUNA_DIR}", "-y"],
        capture_output=True, text=True,
    )
    if _result.returncode != 0:
        raise RuntimeError(
            f"Error al extraer {_archive.name}:\n{_result.stderr}"
        )
    print("✓ Extracción completada")

# ── Verificar estructura extraída ────────────────────────────────────
_csv = LUNA_DIR / "candidates_V2.csv"
_npy_files = list(LUNA_DIR.glob("candidate_*.npy"))
_ok_csv = _csv.exists()
_ok_npy = len(_npy_files) > 0
print(f"  {'✓' if _ok_csv else '✗'} candidates_V2.csv: {_csv.relative_to(PROJECT_ROOT)}")
print(f"  {'✓' if _ok_npy else '✗'} Parches .npy: {len(_npy_files):,} archivos en {LUNA_DIR.relative_to(PROJECT_ROOT)}")
if not _ok_csv or not _ok_npy:
    raise FileNotFoundError(
        "Faltan archivos tras la extracción. Verifica que luna16.7z esté íntegro."
    )
print(f"\n✓ Dataset LUNA16 listo ({len(_npy_files):,} parches)")

# Expert 3 — DenseNet 3D · LUNA16 (Nódulos Pulmonares CT 3D)

**Notebook 100% autosuficiente** para entrenar Expert 3 del pipeline multi-experto.

> **IMPORTANTE — Parches preprocesados requeridos:** Este notebook **NO** trabaja con CTs raw de LUNA16.
> Espera parches `.npy` de 64³ vóxeles **ya preprocesados** por `pre_embeddings.py`, que incluye:
> 1. Extracción de parches 64³ centrados en cada candidato
> 2. Normalización HU a rango [0, 1]
> 3. Zero-centering (resta de `_LUNA_GLOBAL_MEAN ≈ 0.0992`)
>
> Los parches en disco están en rango **[-0.099, 0.901]**, NO en [0, 1].
> Si no tienes los parches, ejecuta primero el pipeline de preprocesamiento o descárgalos de Kaggle.

| Concepto | Detalle |
|---|---|
| **Tarea** | Clasificación binaria de parches 3D CT: nódulo vs. no-nódulo |
| **Arquitectura** | DenseNet 3D from-scratch (~6.7M parámetros) |
| **Loss** | FocalLoss(γ=2, α=0.85) + label smoothing 0.05 |
| **Datos** | LUNA16 — parches `.npy` 64³ preprocesados y zero-centered (rango ≈ [-0.099, 0.901]) |
| **Regularización** | SpatialDropout3d(0.15), Dropout FC(0.4), WD=0.03 |
| **Precisión** | FP16 (mixed precision) + gradient accumulation (batch efectivo=32) |

**Requisitos:** GPU con ≥12 GB VRAM. Parches `.npy` preprocesados en las rutas correctas.

## 1 · Instalación de dependencias

In [ ]:
!pip install pandas scikit-learn tqdm scipy

## 2 · Imports

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import time
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import (
    gaussian_filter,
    map_coordinates,
    rotate as scipy_rotate,
    zoom as scipy_zoom,
)
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from torch.amp import GradScaler
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3 · Configuración de rutas

Ajusta `PROJECT_ROOT` a la raíz de tu workspace. Las demás rutas se derivan automáticamente.

> **Nota:** Las rutas apuntan a parches `.npy` preprocesados por `pre_embeddings.py`, NO a CTs raw.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONFIG — Rutas del proyecto                                        ║
# ║  PROJECT_ROOT viene de la Celda 0 (Setup & Extracción)             ║
# ║                                                                      ║
# ║  ⚠ ESTAS RUTAS APUNTAN A PARCHES PREPROCESADOS (.npy 64³)         ║
# ║  Estructura: todos los .npy PLANOS en datasets/luna16/              ║
# ║  (sin subcarpetas train/val/test — split se hace por CSV)           ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Si ejecutas esta celda de forma aislada, descomenta la siguiente línea:
# PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())

# ── Rutas de datos ──────────────────────────────────────────────────
# Todos los parches .npy están planos en esta carpeta (sin train/val/test)
PATCHES_DIR = PROJECT_ROOT / "datasets" / "luna16"

# CSV con labels (class=0/1) para cada candidato (índice = fila del CSV)
CANDIDATES_V2_CSV = PROJECT_ROOT / "datasets" / "luna16" / "candidates_V2.csv"

# ── Ruta de salida del checkpoint ───────────────────────────────────
# Bug C3 corregido: era "expert_03_vivit_tiny", ahora "expert_03_densenet3d"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "expert_03_densenet3d"
CHECKPOINT_PATH = CHECKPOINT_DIR / "expert3_best.pt"
TRAINING_LOG_PATH = CHECKPOINT_DIR / "expert3_training_log.json"

# ── Verificar que los archivos existen ──────────────────────────────
missing = []
for p, desc in [
    (PATCHES_DIR,       "Directorio de parches .npy (plano)"),
    (CANDIDATES_V2_CSV, "candidates_V2.csv (labels)"),
]:
    exists = p.exists()
    status = "✓" if exists else "✗ NO ENCONTRADO"
    print(f"  {status}  {desc}: {p}")
    if not exists:
        missing.append(desc)

if missing:
    print()
    print("⚠" * 35)
    print("  FALTAN ARCHIVOS REQUERIDOS:")
    for m in missing:
        print(f"    → {m}")
    print()
    print("  Ejecuta primero la Celda 0 (Setup & Extracción) para descomprimir luna16.7z.")
    print("⚠" * 35)
else:
    _npy_count = len(list(PATCHES_DIR.glob("candidate_*.npy")))
    print(f"  ✓  Parches disponibles: {_npy_count:,}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  ✓  Checkpoint dir: {CHECKPOINT_DIR}")

## 4 · Hiperparámetros

Todos los hiperparámetros inline — sin importar `expert3_config.py`.

In [ ]:
# ── Dependencias cruzadas resueltas inline ─────────────────────────
EXPERT_IDS = {"isic": 2, "luna": 3}

# ── Optimizador ────────────────────────────────────────────────────
EXPERT3_LR = 3e-4
EXPERT3_WEIGHT_DECAY = 0.03

# ── Loss ───────────────────────────────────────────────────────────
EXPERT3_FOCAL_GAMMA = 2.0
EXPERT3_FOCAL_ALPHA = 0.85
EXPERT3_LABEL_SMOOTHING = 0.05

# ── Regularización del modelo ──────────────────────────────────────
EXPERT3_DROPOUT_FC = 0.4
EXPERT3_SPATIAL_DROPOUT_3D = 0.15

# ── Batch y entrenamiento ──────────────────────────────────────────
EXPERT3_BATCH_SIZE = 4
EXPERT3_ACCUMULATION_STEPS = 8   # batch efectivo = 4 × 8 = 32
EXPERT3_FP16 = True

# ── Scheduler y stopping ──────────────────────────────────────────
EXPERT3_MAX_EPOCHS = 100
EXPERT3_EARLY_STOPPING_PATIENCE = 20
EXPERT3_EARLY_STOPPING_MONITOR = "val_loss"

# ── Constantes internas ────────────────────────────────────────────
SEED = 42
MIN_DELTA = 0.001   # mejora mínima para early stopping
NUM_WORKERS = 4

# ── Zero-centering global mean — computado en LUNA16 train split ──
# Los parches .npy en disco YA tienen zero-centering aplicado por
# pre_embeddings.py. Su rango real es [-0.099, 0.901], NO [0, 1].
# Esta constante NO se resta en __getitem__ (ya está restada en disco).
# Se usa SOLO en _augment_3d para el clip final tras augmentaciones:
#   np.clip(volume, -_LUNA_GLOBAL_MEAN, 1.0 - _LUNA_GLOBAL_MEAN)
_LUNA_GLOBAL_MEAN: float = 0.09921630471944809

# ── Resumen ────────────────────────────────────────────────────────
CONFIG_SUMMARY = (
    f"Expert 3 (LUNA16 / DenseNet3D): LR={EXPERT3_LR} | WD={EXPERT3_WEIGHT_DECAY} | "
    f"FocalLoss(γ={EXPERT3_FOCAL_GAMMA}, α={EXPERT3_FOCAL_ALPHA}) | "
    f"label_smooth={EXPERT3_LABEL_SMOOTHING} | "
    f"dropout_fc={EXPERT3_DROPOUT_FC} | spatial_drop3d={EXPERT3_SPATIAL_DROPOUT_3D} | "
    f"batch={EXPERT3_BATCH_SIZE} | accum={EXPERT3_ACCUMULATION_STEPS} "
    f"(efectivo={EXPERT3_BATCH_SIZE * EXPERT3_ACCUMULATION_STEPS}) | "
    f"FP16={EXPERT3_FP16} | patience={EXPERT3_EARLY_STOPPING_PATIENCE} | "
    f"max_epochs={EXPERT3_MAX_EPOCHS}"
)
print(CONFIG_SUMMARY)

## 5 · Augmentaciones 3D

Pipeline completo de data augmentation para parches CT 3D:
1. Flips aleatorios (3 ejes, P=0.5 c/u)
2. Rotaciones 3D ±15° (axial, coronal, sagital)
3. Zoom/escalado [0.80, 1.20]
4. Traslación ±4 vóxeles
5. Deformación elástica 3D (P=0.5)
6. Ruido gaussiano + brillo/contraste + blur

In [ ]:
def _random_spatial_shift(volume: np.ndarray, max_shift: int = 4) -> np.ndarray:
    """
    Desplazamiento espacial aleatorio con zero-padding.

    np.roll introducía artefactos: los vóxeles del borde opuesto aparecían
    en el borde desplazado. Con zero-padding, el borde desplazado se rellena
    con 0.0 (equivalente a aire = -1000 HU normalizado).
    """
    D = volume.shape[0]
    for axis in range(3):
        shift = random.randint(-max_shift, max_shift)
        if shift == 0:
            continue
        shifted = np.zeros_like(volume)
        slices_src = [slice(None)] * 3
        slices_dst = [slice(None)] * 3
        if shift > 0:
            slices_src[axis] = slice(0, D - shift)
            slices_dst[axis] = slice(shift, D)
        else:
            slices_src[axis] = slice(-shift, D)
            slices_dst[axis] = slice(0, D + shift)
        shifted[tuple(slices_dst)] = volume[tuple(slices_src)]
        volume = shifted
    return volume


def _augment_3d(volume: np.ndarray) -> np.ndarray:
    """
    Pipeline de data augmentation 3D (7 augmentaciones) para parches CT LUNA16.
    Solo se aplica en split='train'.

    Orden:
      1. Flips aleatorios en los 3 ejes (P=0.5 c/u)
      2. Rotaciones 3D ±15° en los 3 planos (axial, coronal, sagital)
      3. Escalado/Zoom aleatorio [0.80, 1.20] con crop/pad central
      4. Traslación aleatoria ±4 vóxeles (zero-padding)
      5. Deformación elástica 3D (P=0.5, σ∈[1,3], α∈[0,5])
      6. Intensidad: ruido gaussiano + brillo/contraste + blur

    Entrada y salida: ndarray float32 shape (64,64,64).
    """
    D = volume.shape[0]  # 64

    # ── Aug. 2: Flips aleatorios en los 3 ejes (P=0.5 c/u) ────────────
    for axis in range(3):
        if random.random() < 0.5:
            volume = np.flip(volume, axis=axis)

    # ── Aug. 3: Rotaciones 3D ±15° en los 3 planos ─────────────────────
    for axes in [(1, 2), (0, 2), (0, 1)]:
        angle = random.uniform(-15.0, 15.0)
        if abs(angle) > 0.5:
            volume = scipy_rotate(
                volume,
                angle=angle,
                axes=axes,
                reshape=False,
                order=1,
                mode="nearest",
            )

    # ── Aug. 4: Escalado/Zoom aleatorio [0.80, 1.20] ────────────────────
    zoom_factor = random.uniform(0.80, 1.20)
    zoomed = scipy_zoom(volume, zoom_factor, order=1)
    zD = zoomed.shape[0]
    if zD > D:
        start = (zD - D) // 2
        volume = zoomed[start : start + D, start : start + D, start : start + D]
    elif zD < D:
        pad_before = (D - zD) // 2
        pad_after = D - zD - pad_before
        volume = np.pad(
            zoomed,
            ((pad_before, pad_after),) * 3,
            constant_values=0.0,
        )
    else:
        volume = zoomed

    # ── Aug. 5: Traslación aleatoria ±4 vóxeles (zero-padding) ─────────
    volume = _random_spatial_shift(volume, max_shift=4)

    # ── Aug. 6: Deformación elástica 3D (P=0.5) ─────────────────────────
    if random.random() < 0.5:
        sigma = random.uniform(1.0, 3.0)
        alpha = random.uniform(0.0, 5.0)
        shape = volume.shape
        dz = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
        dy = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
        dx = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
        z_g, y_g, x_g = np.meshgrid(
            np.arange(shape[0]),
            np.arange(shape[1]),
            np.arange(shape[2]),
            indexing="ij",
        )
        volume = map_coordinates(
            volume, [z_g + dz, y_g + dy, x_g + dx], order=1, mode="nearest"
        ).astype(np.float32)

    # ── Aug. 7: Augmentación de intensidad ──────────────────────────────

    # 7a. Ruido gaussiano aditivo σ∈[0, 25 HU] normalizado (P=0.5)
    if random.random() < 0.5:
        hu_range = 1400.0
        sigma_noise = random.uniform(0.0, 25.0) / hu_range
        noise = np.random.normal(0.0, sigma_noise, size=volume.shape).astype(
            np.float32
        )
        volume = volume + noise

    # 7b. Ajuste de brillo/contraste
    scale = random.uniform(0.9, 1.1)
    hu_range = 1400.0
    offset_norm = random.uniform(-20.0, 20.0) / hu_range
    volume = volume * scale + offset_norm

    # 7c. Blur gaussiano ligero σ∈[0.1, 0.5] vóxeles (P=0.5)
    if random.random() < 0.5:
        sigma_blur = random.uniform(0.1, 0.5)
        volume = gaussian_filter(volume, sigma=sigma_blur).astype(np.float32)

    volume = np.ascontiguousarray(volume, dtype=np.float32)

    # Clip final para mantener rango físico zero-centered
    volume = np.clip(volume, -_LUNA_GLOBAL_MEAN, 1.0 - _LUNA_GLOBAL_MEAN)

    return volume


print("Augmentaciones 3D definidas ✓")

## 6 · Dataset — LUNA16ExpertDataset

Dataset ligero para entrenamiento del Expert 3. Escanea archivos `.npy` preprocesados (64³, zero-centered) en disco y busca labels en un dict pre-computado (init ~1s vs ~3 min del dataset pesado).

> **Nota:** Los parches `.npy` ya incluyen normalización HU y zero-centering. El dataset NO aplica ninguna normalización adicional — solo carga y opcionalmente aplica augmentation 3D.

In [ ]:
class LUNA16ExpertDataset(Dataset):
    """
    Dataset ligero para entrenamiento del Expert 3 (mode="expert").

    Espera parches .npy de 64³ vóxeles PREPROCESADOS por pre_embeddings.py:
      - Normalización HU a [0, 1] ya aplicada
      - Zero-centering (_LUNA_GLOBAL_MEAN ≈ 0.0992) ya restada
      - Rango real en disco: [-0.099, 0.901]

    NO aplica normalización adicional en __getitem__. La zero-centering
    ya está incorporada en los archivos .npy.

    Escanea los .npy en disco y busca sus labels en un dict
    pre-computado. Resultado: init ~1 segundo vs ~3 minutos.

    Args:
        patches_dir: carpeta con archivos .npy preprocesados del split
        label_map: dict {candidate_index: class_label}
        split: "train", "val" o "test"
        augment_3d: aplicar augmentation 3D (solo en split="train")
    """

    def __init__(
        self,
        patches_dir: str | Path,
        label_map: dict[int, int],
        split: str = "train",
        augment_3d: bool = True,
    ):
        self.patches_dir = Path(patches_dir)
        self.split = split
        self._apply_augment = split == "train" and augment_3d

        # Escanear archivos y mapear labels
        t0 = time.time()
        self.samples: list[tuple[Path, int]] = []
        n_unknown = 0

        for npy_file in sorted(self.patches_dir.glob("candidate_*.npy")):
            try:
                idx = int(npy_file.stem.split("_")[1])
            except (IndexError, ValueError):
                continue
            label = label_map.get(idx, -1)
            if label >= 0:
                self.samples.append((npy_file, label))
            else:
                n_unknown += 1

        elapsed = time.time() - t0
        n_pos = sum(1 for _, c in self.samples if c == 1)
        n_neg = sum(1 for _, c in self.samples if c == 0)

        print(
            f"[LUNA16Expert/{split}] {len(self.samples):,} parches en {elapsed:.2f}s | "
            f"pos={n_pos:,} | neg={n_neg:,} | ratio={n_neg / max(n_pos, 1):.1f}:1"
            f"{f' | {n_unknown} sin label' if n_unknown else ''}"
        )
        if self._apply_augment:
            print(
                f"[LUNA16Expert/{split}] Augmentation 3D ACTIVO: "
                "flip, rotación ±15°, zoom, traslación, deformación elástica, "
                "ruido, brillo/contraste, blur"
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int, str]:
        patch_file, label = self.samples[idx]

        try:
            volume = np.load(patch_file)
        except Exception as e:
            print(f"[LUNA16Expert] Error cargando '{patch_file}': {e}")
            return torch.zeros(1, 64, 64, 64), label, patch_file.stem

        # Augmentation 3D (solo train)
        if self._apply_augment:
            volume = _augment_3d(volume)

        # Convertir a tensor [1, 64, 64, 64]
        volume_t = torch.from_numpy(
            np.ascontiguousarray(volume, dtype=np.float32)
        ).unsqueeze(0)

        return volume_t, label, patch_file.stem


print("LUNA16ExpertDataset definido ✓")

## 7 · FocalLoss (binaria)

FocalLoss de Lin et al. (ICCV 2017). Reduce el gradiente de los ejemplos negativos fáciles y concentra el aprendizaje en los positivos y negativos difíciles.

> **Bug C2 resuelto:** `train_expert3.py` importaba `from losses import FocalLoss` de forma rota. Aquí va inline.

In [ ]:
class FocalLoss(nn.Module):
    """
    FocalLoss (Lin et al., ICCV 2017 — arXiv:1708.02002).

    Reduce el gradiente de los ejemplos negativos fáciles y concentra el
    aprendizaje en los positivos y los negativos difíciles.

    LUNA16:   gamma=2, alpha=0.85

    CORRECCIÓN: alpha=0.85 pondera la clase positiva (nódulos, minoritaria).
    Con ratio 10:1 neg:pos en disco (n_neg/n_total = 13470/14728 ≈ 0.915),
    alpha=0.85 es una aproximación conservadora.

    Uso:
        criterion = FocalLoss(gamma=2, alpha=0.85)
        loss = criterion(logits, labels.float())  # logits: [B], labels: [B] ∈ {0,1}
    """

    def __init__(self, gamma: float = 2.0, alpha: float = 0.85):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


print("FocalLoss definida ✓")

## 8 · Modelo — Expert3DenseNet3D (completo inline)

DenseNet 3D from-scratch (Huang et al., 2017) con convoluciones 3D.

| Componente | Configuración |
|---|---|
| growth_rate | 32 |
| block_layers | [4, 8, 16, 12] |
| init_features | 64 |
| compression | 0.5 |
| Parámetros | ~6.7M |

In [ ]:
class SpatialDropout3d(nn.Module):
    """
    Dropout espacial 3D: desactiva canales completos del feature map.

    Más efectivo para datos volumétricos donde las activaciones son
    espacialmente correlacionadas (ej: tejido pulmonar en CT).

    Args:
        p: probabilidad de desactivar un canal completo. Default: 0.15.
    """

    def __init__(self, p: float = 0.15) -> None:
        super().__init__()
        self.dropout = nn.Dropout3d(p=p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x)


class _DenseLayer(nn.Module):
    """
    Una capa dentro de un DenseBlock 3D (patrón bottleneck).

    Secuencia: BN → ReLU → Conv3d(1×1×1) → BN → ReLU → Conv3d(3×3×3)
    La salida se concatena con la entrada (conectividad densa).
    """

    def __init__(
        self,
        in_features: int,
        growth_rate: int,
        bn_size: int = 4,
    ) -> None:
        super().__init__()
        mid_features = bn_size * growth_rate

        # Bottleneck: 1×1×1
        self.bn1 = nn.BatchNorm3d(in_features)
        self.conv1 = nn.Conv3d(
            in_features, mid_features, kernel_size=1, stride=1, bias=False,
        )

        # Conv 3×3×3
        self.bn2 = nn.BatchNorm3d(mid_features)
        self.conv2 = nn.Conv3d(
            mid_features, growth_rate, kernel_size=3, stride=1, padding=1, bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv1(F.relu(self.bn1(x), inplace=True))
        out = self.conv2(F.relu(self.bn2(out), inplace=True))
        return torch.cat([x, out], dim=1)


class _DenseBlock(nn.Module):
    """
    Bloque denso: secuencia de n_layers DenseLayers con conectividad densa.
    x_l = H_l([x_0, x_1, ..., x_{l-1}])
    """

    def __init__(
        self,
        n_layers: int,
        in_features: int,
        growth_rate: int,
        bn_size: int = 4,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(n_layers):
            self.layers.append(
                _DenseLayer(
                    in_features=in_features + i * growth_rate,
                    growth_rate=growth_rate,
                    bn_size=bn_size,
                )
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


class _Transition(nn.Module):
    """
    Capa de transición entre DenseBlocks.
    BN → ReLU → Conv3d(1×1×1) → AvgPool3d(2×2×2)
    """

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.bn = nn.BatchNorm3d(in_features)
        self.conv = nn.Conv3d(
            in_features, out_features, kernel_size=1, stride=1, bias=False,
        )
        self.pool = nn.AvgPool3d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv(F.relu(self.bn(x), inplace=True))
        return self.pool(out)


class Expert3DenseNet3D(nn.Module):
    """
    DenseNet 3D para clasificación binaria de parches 3D CT (LUNA16).

    Implementación from-scratch de DenseNet (Huang et al., 2017) con
    convoluciones 3D, diseñada para volúmenes CT médicos.

    Entrada:  [B, 1, 64, 64, 64] — parche CT monocanal normalizado
    Salida:   [B, 2] — logits para {no-nódulo, nódulo}
    """

    def __init__(
        self,
        in_channels: int = 1,
        num_classes: int = 2,
        growth_rate: int = 32,
        block_layers: list[int] | None = None,
        init_features: int = 64,
        bn_size: int = 4,
        compression: float = 0.5,
        spatial_dropout_p: float = 0.15,
        fc_dropout_p: float = 0.4,
    ) -> None:
        super().__init__()

        if block_layers is None:
            block_layers = [4, 8, 16, 12]

        self.growth_rate = growth_rate
        self.block_layers = block_layers

        # ── Stem: Conv3d(7×7×7) + BN + ReLU + MaxPool + SpatialDropout ──
        self.stem = nn.Sequential(
            OrderedDict(
                [
                    (
                        "conv0",
                        nn.Conv3d(
                            in_channels, init_features,
                            kernel_size=7, stride=2, padding=3, bias=False,
                        ),
                    ),
                    ("norm0", nn.BatchNorm3d(init_features)),
                    ("relu0", nn.ReLU(inplace=True)),
                    ("pool0", nn.MaxPool3d(kernel_size=3, stride=2, padding=1)),
                    ("spatial_dropout", SpatialDropout3d(p=spatial_dropout_p)),
                ]
            )
        )

        # ── Dense blocks + Transition layers ────────────────────────────
        n_features = init_features
        self.dense_blocks = nn.ModuleList()
        self.transitions = nn.ModuleList()

        for i, n_layers in enumerate(block_layers):
            block = _DenseBlock(
                n_layers=n_layers,
                in_features=n_features,
                growth_rate=growth_rate,
                bn_size=bn_size,
            )
            self.dense_blocks.append(block)
            n_features = n_features + n_layers * growth_rate

            # Transition después de cada bloque excepto el último
            if i < len(block_layers) - 1:
                out_features = int(n_features * compression)
                transition = _Transition(n_features, out_features)
                self.transitions.append(transition)
                n_features = out_features

        # ── BN final ────────────────────────────────────────────────────
        self.final_bn = nn.BatchNorm3d(n_features)

        # ── Global Average Pooling ──────────────────────────────────────
        self.avgpool = nn.AdaptiveAvgPool3d(1)

        # ── Cabeza clasificadora con dropout ────────────────────────────
        self.classifier = nn.Sequential(
            nn.Dropout(p=fc_dropout_p),
            nn.Linear(n_features, num_classes),
        )

        # ── Inicialización de pesos ─────────────────────────────────────
        self._init_weights()

    def _init_weights(self) -> None:
        """Inicialización: Kaiming para Conv3d, constante para BN, normal para Linear."""
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Stem: [B,1,64,64,64] → [B,64,16,16,16]
        x = self.stem(x)

        # Dense blocks + transitions
        for i, block in enumerate(self.dense_blocks):
            x = block(x)
            if i < len(self.transitions):
                x = self.transitions[i](x)

        # Final BN + ReLU
        x = F.relu(self.final_bn(x), inplace=True)

        # Global average pooling + flatten
        x = self.avgpool(x)
        x = x.flatten(1)

        # Clasificador
        x = self.classifier(x)
        return x

    def count_parameters(self) -> int:
        """Retorna el número total de parámetros entrenables."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_all_parameters(self) -> int:
        """Retorna el número total de parámetros (entrenables + congelados)."""
        return sum(p.numel() for p in self.parameters())


# ── Alias de compatibilidad ─────────────────────────────────────────
Expert3MC318 = Expert3DenseNet3D


# ── Verificación rápida ─────────────────────────────────────────────
_test_model = Expert3DenseNet3D(
    spatial_dropout_p=EXPERT3_SPATIAL_DROPOUT_3D,
    fc_dropout_p=EXPERT3_DROPOUT_FC,
    num_classes=2,
)
print(f"Expert3DenseNet3D definido ✓ — {_test_model.count_parameters():,} parámetros")
del _test_model

## 9 · DataLoaders

Carga CSV → label_map, crea datasets para cada split, construye DataLoaders.

In [ ]:
def build_dataloaders(
    patches_dir: Path,
    candidates_csv: Path,
    batch_size: int,
    num_workers: int,
    val_frac: float = 0.10,
    test_frac: float = 0.10,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Construye DataLoaders para train, val y test del Expert 3.

    Compatible con estructura plana: todos los parches .npy en patches_dir
    (sin subcarpetas train/val/test). El split se determina:
      1. Si candidates_V2.csv tiene columna 'split': usa esos valores.
      2. Si no: split determinístico 80/10/10 con SEED fijo.

    Args:
        patches_dir:    directorio con todos los candidate_*.npy (estructura plana)
        candidates_csv: CSV con al menos columna 'class' (0/1); opcionalmente 'split'
        batch_size:     tamaño de mini-batch
        num_workers:    workers del DataLoader
        val_frac:       fracción para val (solo si no hay columna 'split')
        test_frac:      fracción para test (solo si no hay columna 'split')
    """
    # ── Cargar CSV ──────────────────────────────────────────────────
    t0 = time.time()
    df = pd.read_csv(candidates_csv)
    elapsed = time.time() - t0
    n_pos = (df["class"] == 1).sum()
    n_neg = (df["class"] == 0).sum()
    print(
        f"[DataLoader] CSV cargado: {candidates_csv.name} | "
        f"{len(df):,} candidatos (pos={n_pos:,}, neg={n_neg:,}) | {elapsed:.2f}s"
    )

    # ── Determinar split ────────────────────────────────────────────
    if "split" in df.columns:
        print("[DataLoader] Columna 'split' detectada en CSV — usando splits predefinidos")
        train_idx = df[df["split"] == "train"].index
        val_idx   = df[df["split"] == "val"].index
        test_idx  = df[df["split"] == "test"].index
    else:
        print(f"[DataLoader] Sin columna 'split' — generando split determinístico "
              f"(train={1-val_frac-test_frac:.0%} / val={val_frac:.0%} / test={test_frac:.0%}, seed={SEED})")
        rng = np.random.default_rng(SEED)
        all_idx = rng.permutation(np.array(df.index))
        n = len(all_idx)
        n_test  = int(test_frac * n)
        n_val   = int(val_frac * n)
        n_train = n - n_val - n_test
        train_idx = all_idx[:n_train]
        val_idx   = all_idx[n_train : n_train + n_val]
        test_idx  = all_idx[n_train + n_val :]

    # ── Construir label maps por split ──────────────────────────────
    all_labels = dict(zip(df.index.astype(int), df["class"].astype(int)))
    train_label_map = {int(i): all_labels[int(i)] for i in train_idx}
    val_label_map   = {int(i): all_labels[int(i)] for i in val_idx}
    test_label_map  = {int(i): all_labels[int(i)] for i in test_idx}

    # ── Crear datasets (todos usan el mismo directorio plano) ───────
    train_ds = LUNA16ExpertDataset(
        patches_dir=patches_dir,
        label_map=train_label_map,
        split="train",
        augment_3d=True,
    )
    val_ds = LUNA16ExpertDataset(
        patches_dir=patches_dir,
        label_map=val_label_map,
        split="val",
        augment_3d=False,
    )
    test_ds = LUNA16ExpertDataset(
        patches_dir=patches_dir,
        label_map=test_label_map,
        split="test",
        augment_3d=False,
    )

    # ── Crear DataLoaders ───────────────────────────────────────────
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )

    # ── Resumen ──────────────────────────────────────────────────────
    print("=" * 70)
    print("  RESUMEN — DataLoaders Expert 3 (LUNA16)")
    print("=" * 70)
    for name, ds, loader in [
        ("train", train_ds, train_loader),
        ("val", val_ds, val_loader),
        ("test", test_ds, test_loader),
    ]:
        n_pos = sum(1 for _, c in ds.samples if c == 1)
        n_neg = sum(1 for _, c in ds.samples if c == 0)
        ratio = n_neg / max(n_pos, 1)
        aug_status = "SÍ" if ds._apply_augment else "NO"
        print(
            f"  {name:5s}: {len(ds):>6,} muestras | "
            f"pos={n_pos:>5,} | neg={n_neg:>5,} | "
            f"ratio={ratio:>5.1f}:1 | "
            f"batches={len(loader):>4,} | aug={aug_status}"
        )
    print("=" * 70)

    return train_loader, val_loader, test_loader


# ── Construir DataLoaders ───────────────────────────────────────────
train_loader, val_loader, test_loader = build_dataloaders(
    patches_dir=PATCHES_DIR,
    candidates_csv=CANDIDATES_V2_CSV,
    batch_size=EXPERT3_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

## 10 · Funciones de entrenamiento y validación

In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Fija todas las semillas para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[Seed] Semillas fijadas a {seed}")


def log_vram(tag: str = "") -> None:
    """Imprime uso actual de VRAM si hay GPU disponible."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(
            f"[VRAM{' ' + tag if tag else ''}] "
            f"Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB"
        )


def enable_gradient_checkpointing(model: Expert3DenseNet3D) -> bool:
    """
    Habilita gradient checkpointing en los dense blocks del DenseNet 3D.

    Libera activaciones intermedias y las recalcula en el backward,
    reduciendo uso de VRAM significativamente.
    """
    try:
        from torch.utils.checkpoint import checkpoint

        if not hasattr(model, "dense_blocks") or not model.dense_blocks:
            print("[GradCheckpoint] Modelo sin 'dense_blocks'. No aplicable.")
            return False

        checkpointed_count = 0
        for idx, block in enumerate(model.dense_blocks):
            def make_checkpointed_forward(original_fwd):
                def checkpointed_forward(x):
                    return checkpoint(original_fwd, x, use_reentrant=False)
                return checkpointed_forward

            block.forward = make_checkpointed_forward(block.forward)
            checkpointed_count += 1

        print(
            f"[GradCheckpoint] Gradient checkpointing HABILITADO en "
            f"{checkpointed_count} dense blocks"
        )
        return True
    except Exception as e:
        print(f"[GradCheckpoint] No se pudo habilitar: {e}")
        return False


class EarlyStopping:
    """Early stopping por val_loss con patience configurable."""

    def __init__(self, patience: int, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                return True
            return False


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    accumulation_steps: int,
    use_fp16: bool,
) -> float:
    """
    Ejecuta una época de entrenamiento con gradient accumulation y FP16.

    Returns:
        Loss promedio de la época.
    """
    model.train()
    total_loss = 0.0
    n_batches = 0

    optimizer.zero_grad()

    for batch_idx, (volumes, labels, _stems) in enumerate(loader):
        volumes = volumes.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)

        # Forward con autocast FP16
        with torch.amp.autocast(device_type=device.type, enabled=use_fp16):
            logits = model(volumes)  # [B, 2]
            # Label smoothing: {0,1} → {0.025, 0.975}
            labels_smooth = (
                labels * (1 - EXPERT3_LABEL_SMOOTHING) + EXPERT3_LABEL_SMOOTHING / 2.0
            )
            loss = criterion(logits[:, 1], labels_smooth)
            loss = loss / accumulation_steps

        # Backward con GradScaler
        scaler.scale(loss).backward()

        # Optimizer step cada accumulation_steps batches
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        n_batches += 1

    # Flush de gradientes residuales
    if n_batches % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_fp16: bool,
) -> dict:
    """
    Ejecuta validación y calcula métricas.

    Returns:
        dict con keys: val_loss, val_f1_macro, val_auc, confusion_matrix
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0

    all_labels = []
    all_probs = []
    all_preds = []

    for batch_idx, (volumes, labels, _stems) in enumerate(loader):
        volumes = volumes.to(device, non_blocking=True)
        labels_float = labels.float().to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_fp16):
            logits = model(volumes)  # [B, 2]
            loss = criterion(logits[:, 1], labels_float)

        total_loss += loss.item()
        n_batches += 1

        # Probabilidades y predicciones
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

    avg_loss = total_loss / max(n_batches, 1)

    # Métricas
    labels_arr = np.array(all_labels)
    preds_arr = np.array(all_preds)
    probs_arr = np.array(all_probs)

    # F1 Macro
    if len(np.unique(labels_arr)) >= 2:
        f1_macro = f1_score(labels_arr, preds_arr, average="macro", zero_division=0)
    else:
        f1_macro = 0.0

    # AUC-ROC
    if len(np.unique(labels_arr)) >= 2:
        try:
            auc = roc_auc_score(labels_arr, probs_arr)
        except ValueError:
            auc = 0.0
    else:
        auc = 0.0

    # Confusion matrix
    cm = confusion_matrix(labels_arr, preds_arr, labels=[0, 1])

    return {
        "val_loss": avg_loss,
        "val_f1_macro": f1_macro,
        "val_auc": auc,
        "confusion_matrix": cm.tolist(),
    }


print("Funciones de entrenamiento y validación definidas ✓")

## 11 · Training loop

Fase única con early stopping por `val_loss`. Guarda el mejor checkpoint.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  ENTRENAMIENTO — Expert 3 (DenseNet3D / LUNA16)
# ══════════════════════════════════════════════════════════════════════

set_seed(SEED)

# ── Dispositivo ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Expert3] Dispositivo: {device}")
if device.type == "cpu":
    print("⚠ Entrenando en CPU — será muy lento. Se recomienda GPU ≥12 GB VRAM.")

use_fp16 = EXPERT3_FP16 and device.type == "cuda"
if not use_fp16 and EXPERT3_FP16:
    print("[Expert3] FP16 desactivado (no hay GPU). Usando FP32 en CPU.")

# ── Modelo ─────────────────────────────────────────────────────────
model = Expert3DenseNet3D(
    spatial_dropout_p=EXPERT3_SPATIAL_DROPOUT_3D,
    fc_dropout_p=EXPERT3_DROPOUT_FC,
    num_classes=2,
).to(device)

n_params = model.count_parameters()
print(f"[Expert3] Modelo DenseNet3D creado: {n_params:,} parámetros entrenables")
log_vram("post-model")

# ── Gradient checkpointing ─────────────────────────────────────────
if device.type == "cuda":
    enable_gradient_checkpointing(model)

# ── Loss ───────────────────────────────────────────────────────────
criterion = FocalLoss(gamma=EXPERT3_FOCAL_GAMMA, alpha=EXPERT3_FOCAL_ALPHA)
print(f"[Expert3] Loss: FocalLoss(gamma={EXPERT3_FOCAL_GAMMA}, alpha={EXPERT3_FOCAL_ALPHA})")

# ── Optimizer ──────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=EXPERT3_LR,
    weight_decay=EXPERT3_WEIGHT_DECAY,
    betas=(0.9, 0.999),
)
print(f"[Expert3] Optimizer: AdamW(lr={EXPERT3_LR}, wd={EXPERT3_WEIGHT_DECAY})")

# ── Scheduler ──────────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6,
)
print("[Expert3] Scheduler: CosineAnnealingWarmRestarts(T_0=15, T_mult=2, eta_min=1e-6)")

# ── GradScaler para FP16 ───────────────────────────────────────────
scaler = GradScaler(device=device.type, enabled=use_fp16)

# ── Early stopping ─────────────────────────────────────────────────
early_stopping = EarlyStopping(
    patience=EXPERT3_EARLY_STOPPING_PATIENCE,
    min_delta=MIN_DELTA,
)
print(
    f"[Expert3] EarlyStopping: monitor={EXPERT3_EARLY_STOPPING_MONITOR}, "
    f"patience={EXPERT3_EARLY_STOPPING_PATIENCE}, min_delta={MIN_DELTA}"
)

# ── Training loop ──────────────────────────────────────────────────
best_val_loss = float("inf")
training_log = []

print(f"\n{'=' * 70}")
print(f"  INICIO DE ENTRENAMIENTO — Expert 3 (DenseNet3D / LUNA16)")
print(
    f"  Épocas máx: {EXPERT3_MAX_EPOCHS} | Batch efectivo: "
    f"{EXPERT3_BATCH_SIZE}×{EXPERT3_ACCUMULATION_STEPS}="
    f"{EXPERT3_BATCH_SIZE * EXPERT3_ACCUMULATION_STEPS}"
)
print(f"  FP16: {use_fp16} | Accumulation: {EXPERT3_ACCUMULATION_STEPS}")
print(f"{'=' * 70}\n")

for epoch in range(EXPERT3_MAX_EPOCHS):
    epoch_start = time.time()

    # ── Train ──────────────────────────────────────────────────
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        accumulation_steps=EXPERT3_ACCUMULATION_STEPS,
        use_fp16=use_fp16,
    )

    # ── Validation ─────────────────────────────────────────────
    val_results = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        use_fp16=use_fp16,
    )

    # ── Scheduler step ─────────────────────────────────────────
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # ── Log de época ───────────────────────────────────────────
    epoch_time = time.time() - epoch_start
    val_loss = val_results["val_loss"]
    val_f1 = val_results["val_f1_macro"]
    val_auc = val_results["val_auc"]
    cm = val_results["confusion_matrix"]

    is_best = val_loss < best_val_loss - MIN_DELTA

    print(
        f"[Epoch {epoch + 1:3d}/{EXPERT3_MAX_EPOCHS}] "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1:.4f} | val_auc={val_auc:.4f} | "
        f"lr={current_lr:.2e} | time={epoch_time:.1f}s"
        f"{'  ★ BEST' if is_best else ''}"
    )
    print(
        f"         Confusion Matrix: "
        f"TN={cm[0][0]:>5} FP={cm[0][1]:>5} | "
        f"FN={cm[1][0]:>5} TP={cm[1][1]:>5}"
    )
    log_vram(f"epoch-{epoch + 1}")

    # ── Guardar log de métricas ────────────────────────────────
    epoch_log = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1,
        "val_auc": val_auc,
        "confusion_matrix": cm,
        "lr": current_lr,
        "epoch_time_s": round(epoch_time, 1),
        "is_best": is_best,
    }
    training_log.append(epoch_log)

    # ── Guardar mejor checkpoint ───────────────────────────────
    if is_best:
        best_val_loss = val_loss
        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_auc": val_auc,
            "config": {
                "lr": EXPERT3_LR,
                "weight_decay": EXPERT3_WEIGHT_DECAY,
                "focal_gamma": EXPERT3_FOCAL_GAMMA,
                "focal_alpha": EXPERT3_FOCAL_ALPHA,
                "label_smoothing": EXPERT3_LABEL_SMOOTHING,
                "dropout_fc": EXPERT3_DROPOUT_FC,
                "spatial_dropout_3d": EXPERT3_SPATIAL_DROPOUT_3D,
                "batch_size": EXPERT3_BATCH_SIZE,
                "accumulation_steps": EXPERT3_ACCUMULATION_STEPS,
                "fp16": EXPERT3_FP16,
                "n_params": n_params,
                "seed": SEED,
            },
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"  → Checkpoint guardado: {CHECKPOINT_PATH}")

    # ── Guardar training log ───────────────────────────────────
    with open(TRAINING_LOG_PATH, "w") as f:
        json.dump(training_log, f, indent=2)

    # ── Early stopping ─────────────────────────────────────────
    if early_stopping.step(val_loss):
        print(
            f"\n[EarlyStopping] Detenido en época {epoch + 1}. "
            f"val_loss no mejoró en {EXPERT3_EARLY_STOPPING_PATIENCE} épocas. "
            f"Mejor val_loss: {best_val_loss:.4f}"
        )
        break

# ── Resumen final ──────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"  ENTRENAMIENTO FINALIZADO — Expert 3 (DenseNet3D / LUNA16)")
print(f"  Mejor val_loss: {best_val_loss:.4f}")
if training_log:
    best_epoch = min(training_log, key=lambda x: x["val_loss"])
    print(
        f"  Mejor época: {best_epoch['epoch']} | "
        f"F1 Macro: {best_epoch['val_f1_macro']:.4f} | "
        f"AUC: {best_epoch['val_auc']:.4f}"
    )
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Training log: {TRAINING_LOG_PATH}")
print(f"{'=' * 70}")